# DateTime Hell: Timezones, Mixed Formats & Off-By-One

Datetime columns are the most treacherous dtype in pandas. Ambiguous formats silently
swap day and month, mixing timezone-naive and timezone-aware Series raises cryptic errors,
and resampling with the wrong `closed` boundary attributes an entire day's revenue to the
wrong date. This notebook walks through the three deadliest pitfalls and a safe pipeline
pattern to avoid them.

In [ ]:
!pip install pandas numpy matplotlib --quiet

## Generate Messy Synthetic Timestamps

Real-world CSV ingestion delivers timestamps in every conceivable format. We simulate
a column that mixes ISO, European day-first, US long-form, epoch integers, and `None`.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

raw_timestamps = [
    "2024-01-15 08:30:00",       # ISO — unambiguous
    "2024-03-07 14:00:00",       # ISO — unambiguous
    "01/02/2024 09:00:00",       # Ambiguous! Is this Jan 2 or Feb 1?
    "05/03/2024 11:30:00",       # Ambiguous! May 3 or Mar 5?
    "15/01/2024 10:00:00",       # European day-first (only valid as day-first)
    "Mar 20, 2024 3:00 PM",      # US long-form
    1706745600,                   # Unix epoch (2024-02-01 00:00:00 UTC)
    None,                        # Missing value
    "2024-06-30T23:59:59+02:00", # ISO with timezone offset
    "2024-12-25",                # Date only, no time
]

df = pd.DataFrame({
    "raw_ts": raw_timestamps,
    "revenue": np.round(np.random.uniform(10, 500, size=len(raw_timestamps)), 2),
})

print("Raw data as ingested:")
display(df)

## Pitfall 1: `pd.to_datetime` Silent Failures

Calling `pd.to_datetime()` on mixed-format data will **guess** the format for each row.
For ambiguous dates like `01/02/2024`, pandas defaults to **month-first** (US convention).
If your data is European (day-first), the day and month get silently swapped — no error,
no warning, just wrong data.

In [ ]:
# Naive approach — let pandas guess everything
# We need to separate string and non-string values to demonstrate the issue
string_dates = ["01/02/2024 09:00:00", "05/03/2024 11:30:00", "15/01/2024 10:00:00"]

# Default: month-first (US)
parsed_us = pd.to_datetime(string_dates)
print("Default (month-first) parsing:")
for raw, parsed in zip(string_dates, parsed_us):
    print(f"  {raw!r:>30}  ->  {parsed}")

# With dayfirst=True (European)
parsed_eu = pd.to_datetime(string_dates, dayfirst=True)
print("\ndayfirst=True parsing:")
for raw, parsed in zip(string_dates, parsed_eu):
    print(f"  {raw!r:>30}  ->  {parsed}")

print("\n'01/02/2024' parsed as Jan 2 (US) vs Feb 1 (EU) — completely different dates!")
print("'05/03/2024' parsed as May 3 (US) vs Mar 5 (EU) — wrong month, wrong quarter!")

In [ ]:
# The errors='coerce' trap: it hides bad data as NaT instead of failing
bad_data = ["2024-01-15", "not_a_date", "2024-13-01", "2024-02-30"]

coerced = pd.to_datetime(bad_data, errors="coerce")
print("errors='coerce' output:")
for raw, parsed in zip(bad_data, coerced):
    print(f"  {raw!r:>20}  ->  {parsed}")

print(f"\nNaT count: {coerced.isna().sum()} out of {len(coerced)}")
print("Those NaTs are invisible data loss if you don't check for them!")

# DEFENSE: always count NaTs after coercion
nat_count = coerced.isna().sum()
if nat_count > 0:
    print(f"\nWARNING: {nat_count} values could not be parsed and became NaT.")

## Pitfall 2: Timezone-Naive vs Timezone-Aware Mixing

Pandas refuses to compare, merge, or concatenate timezone-naive and timezone-aware
Series. This typically blows up at merge time, deep in your pipeline, when it's
expensive to debug.

In [ ]:
# Table A: timestamps without timezone (naive — assumed UTC by convention)
table_a = pd.DataFrame({
    "event_time": pd.to_datetime(["2024-06-15 12:00", "2024-06-15 18:00"]),
    "event": ["login", "purchase"],
})

# Table B: timestamps with explicit timezone (aware)
table_b = pd.DataFrame({
    "event_time": pd.to_datetime(["2024-06-15 12:00", "2024-06-15 18:00"]).tz_localize("US/Eastern"),
    "source": ["web", "mobile"],
})

print(f"Table A timezone: {table_a['event_time'].dt.tz}  (naive)")
print(f"Table B timezone: {table_b['event_time'].dt.tz}  (aware)")

# Attempting to merge raises TypeError
try:
    table_a.merge(table_b, on="event_time")
except TypeError as e:
    print(f"\nTypeError: {e}")
    print("Cannot merge naive and aware timestamps!")

In [ ]:
# DEFENSE: normalize everything to UTC
# tz_localize: "this naive timestamp IS in UTC" (assigns timezone, no conversion)
# tz_convert:  "convert this aware timestamp TO UTC" (shifts the clock)

table_a_utc = table_a.copy()
table_a_utc["event_time"] = table_a_utc["event_time"].dt.tz_localize("UTC")

table_b_utc = table_b.copy()
table_b_utc["event_time"] = table_b_utc["event_time"].dt.tz_convert("UTC")

print("After normalization to UTC:")
print(f"Table A: {table_a_utc['event_time'].dt.tz}")
print(f"Table B: {table_b_utc['event_time'].dt.tz}")
print(f"\nTable A times: {table_a_utc['event_time'].tolist()}")
print(f"Table B times: {table_b_utc['event_time'].tolist()}")
print("\nNote: 12:00 Eastern = 16:00 UTC — the times no longer match (correctly!)")

# Now the merge works
merged = table_a_utc.merge(table_b_utc, on="event_time", how="outer")
display(merged)

## Pitfall 3: Off-By-One in Resampling

When resampling time series to daily granularity, the `closed` and `label` parameters
determine which day a timestamp belongs to. The defaults can attribute late-night
transactions to the wrong calendar day.

In [ ]:
import matplotlib.pyplot as plt

# Generate hourly revenue data across 3 days
hours = pd.date_range("2024-03-01", periods=72, freq="h")
np.random.seed(42)
revenue = np.random.uniform(100, 1000, size=72)

# Spike revenue at 23:00 on March 1st to make the off-by-one visible
revenue[23] = 50_000  # 2024-03-01 23:00 — should be March 1st revenue

ts = pd.DataFrame({"timestamp": hours, "revenue": revenue}).set_index("timestamp")

# Default resample: closed='left', label='left'
daily_default = ts.resample("D").sum()

# Business convention: day runs 00:00-23:59, so closed='left' is correct here.
# But what if someone uses offset='1h' thinking it helps?
daily_offset = ts.resample("D", offset="1h").sum()

print("Default resample (correct — day = 00:00 to 23:59):")
display(daily_default)

print("\nWith offset='1h' (WRONG — day starts at 01:00, the 23:00 spike shifts):")
display(daily_offset)

print(f"\nThe $50,000 spike at 2024-03-01 23:00:")
print(f"  Default: attributed to March 1st ({daily_default.loc['2024-03-01', 'revenue']:.0f})")
print(f"  Offset:  attributed to March 1st ({daily_offset.iloc[0]['revenue']:.0f}) — spike may shift!")
print("\nAlways verify which bucket boundary events fall into.")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

daily_default.plot.bar(ax=axes[0], legend=False, color="steelblue")
axes[0].set_title("Default resample (closed='left')")
axes[0].set_ylabel("Daily Revenue ($)")
axes[0].tick_params(axis='x', rotation=0)

daily_offset.plot.bar(ax=axes[1], legend=False, color="coral")
axes[1].set_title("With offset='1h' (shifted buckets)")
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## The Safe DateTime Pipeline

A four-step pattern that prevents all three pitfalls.

In [ ]:
def safe_parse_timestamps(series, expected_format=None, source_tz=None):
    """
    Parse a raw timestamp column safely.

    1. Parse with explicit format (no guessing).
    2. Report any NaTs from coercion.
    3. Localize to source timezone.
    4. Convert to UTC for storage.
    """
    # Step 1: parse with explicit format
    if expected_format:
        parsed = pd.to_datetime(series, format=expected_format, errors="coerce")
    else:
        parsed = pd.to_datetime(series, errors="coerce")

    # Step 2: audit NaTs
    n_nat = parsed.isna().sum()
    n_original_na = series.isna().sum()
    n_coerced = n_nat - n_original_na
    if n_coerced > 0:
        print(f"WARNING: {n_coerced} values failed to parse and became NaT")
        bad_idx = parsed.isna() & series.notna()
        print(f"  Failed values: {series[bad_idx].tolist()[:5]}")

    # Step 3: localize to source timezone
    if source_tz and parsed.dt.tz is None:
        parsed = parsed.dt.tz_localize(source_tz)

    # Step 4: convert to UTC for storage
    if parsed.dt.tz is not None:
        parsed = parsed.dt.tz_convert("UTC")

    return parsed


# Example: parse European-format timestamps
eu_data = pd.Series(["15/01/2024 10:00", "28/02/2024 14:30", "invalid", None])
result = safe_parse_timestamps(eu_data, expected_format="%d/%m/%Y %H:%M", source_tz="Europe/Rome")
print("\nParsed result (UTC):")
print(result.tolist())

## Takeaways

| Pitfall | Symptom | Safe Pattern |
| :--- | :--- | :--- |
| Ambiguous date format (`01/02`) | Day and month silently swapped | Always use `format=` with explicit pattern |
| `errors='coerce'` hides failures | Bad data becomes `NaT` invisibly | Count NaTs before and after parsing |
| Mixing naive and aware timestamps | `TypeError` at merge time | `tz_localize` naive data immediately, store everything in UTC |
| Wrong resample boundaries | Revenue attributed to wrong day | Verify `closed=` and `label=` match your business definition of "day" |

**Golden rule:** parse with an explicit format, localize immediately, store in UTC, convert
to local timezone only at display time.